## Interactive Entity Mapping for PII Evaluation

This notebook is a comprehensive tutorial on `CanonicalMapper` — the two-phase
entity alignment tool for PII evaluation.

**Topics covered:**
1. Why entity mapping is needed
2. Two-phase workflow: Identify (5 tiers) → Project (majority-vote depth)
3. All six issue types with concrete examples
4. Majority-vote depth auto-discovery
5. Projection rules: exact, trivial, ambiguous, cross-branch, unresolved
6. Programmatic resolution with `map()`
7. Interactive resolution with `resolve_interactively()`
8. Eval-surface locking for multi-model comparison
9. EntityHierarchy customisation
10. Getting the final mapped DataFrame


In [ ]:
from collections import Counter
from pathlib import Path
from pprint import pprint

from presidio_analyzer import AnalyzerEngine, PatternRecognizer

from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import (
    CanonicalMapper,
    EntityHierarchy,
    IncompleteMapping,
    IssueType,
    IssueSeverity,
)
from presidio_evaluator.evaluation import SpanEvaluator
from presidio_evaluator.models import PresidioAnalyzerWrapper


## 1. Why Entity Mapping is Needed

When evaluating a PII detection model, you are comparing:
- **Dataset entities**: ground truth labels (e.g., `STREET_ADDRESS`, `FIRST_NAME`, `GPE`)
- **Model entities**: what the model can detect (e.g., `LOCATION`, `PERSON`, `NRP`)

These often use **different naming conventions and hierarchy depths**:

| Dataset Entity | Model Entity | Notes |
|----------------|--------------|-------|
| `STREET_ADDRESS` | `LOCATION` | `STREET_ADDRESS` is depth-3 (`PII→LOCATION→ADDRESS`); Presidio predicts depth-2 `LOCATION` |
| `FIRST_NAME` | `PERSON` | `FIRST_NAME` is depth-4; Presidio predicts depth-2 `PERSON` |
| `GPE` | `LOCATION` | Both refer to geopolitical entities, different names |

`CanonicalMapper` resolves these mismatches automatically in two phases:
1. **Identify** — match each raw label to a canonical entity using 5 tiers
2. **Project** — project canonical entities onto the canonical surface at majority-vote depth


In [2]:
# Load dataset
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))
print(f"Loaded {len(dataset)} samples")

# Show dataset entities
dataset_entities = Counter()
for sample in dataset:
    for span in sample.spans:
        dataset_entities[span.entity_type] += 1

print(f"\nDataset has {len(dataset_entities)} entity types:")
pprint(dict(dataset_entities.most_common(10)), compact=True)

tokenizing input:   0%|          | 1/1500 [00:00<04:51,  5.15it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1500/1500 [00:04<00:00, 339.24it/s]

Loaded 1500 samples

Dataset has 17 entity types:
{'AGE': 74,
 'CREDIT_CARD': 136,
 'DATE_TIME': 119,
 'GPE': 411,
 'NRP': 55,
 'ORGANIZATION': 250,
 'PERSON': 857,
 'PHONE_NUMBER': 92,
 'STREET_ADDRESS': 598,
 'TITLE': 92}


In [3]:
# Load Presidio Analyzer
analyzer = AnalyzerEngine(default_score_threshold=0.4)

# Let's add a custom entity recognizer 
# with an unknown entity type to show how the mapping works
composers_recognizer = PatternRecognizer(supported_entity="COMPOSER", 
                                         deny_list=["A", "Mo", "Bee", "Ba"])

analyzer.registry.add_recognizer(composers_recognizer)

print(f"Presidio supports {len(analyzer.get_supported_entities('en'))} entity types:")
pprint(sorted(analyzer.get_supported_entities("en")), compact=True)

Presidio supports 20 entity types:
['COMPOSER', 'CREDIT_CARD', 'CRYPTO', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE',
 'IP_ADDRESS', 'LOCATION', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'NRP', 'PERSON',
 'PHONE_NUMBER', 'UK_NHS', 'URL', 'US_BANK_NUMBER', 'US_DRIVER_LICENSE',
 'US_ITIN', 'US_PASSPORT', 'US_SSN']


## 2. Two-Phase Workflow: Identify → Project

### Phase 1: Identification (5 tiers, in order)

| Tier | Example | Notes |
|------|---------|-------|
| `EXACT` | `EMAIL_ADDRESS` → `EMAIL_ADDRESS` | Label is already a canonical name or known alias |
| `COUNTRY` | `US_PASSPORT` → `PASSPORT` | Strip country prefix (`US_`, `AU_`, etc.) |
| `COUNTRY_FALLBACK` | `GERMANY_PASSPORT_NUMBER` → `PASSPORT` | Strip country + `_NUMBER` suffix |
| `FUZZY` | `EMAILADRES` → `EMAIL_ADDRESS` | Fuzzy string match above threshold (default 0.80) |
| `UNRESOLVED` | `XYZZY_UNKNOWN` | No match found — must be manually mapped or suppressed |

BIO tags (`B-PERSON`, `I-PERSON`) are automatically stripped before matching.

### Phase 2: Projection

After identification, each canonical entity is projected onto the **canonical surface** —
the set of entities at the **majority-vote depth** computed from the dataset annotations.

Projection rules:
| Rule | Issue Type | Notes |
|------|------------|-------|
| Canonical is already on the canonical surface | `EXACT` projection | No change |
| Canonical is a descendant — one sole ancestor on canonical surface | `TRIVIAL` (auto-fixed, INFO) | Collapsed upward |
| Canonical is a depth-2 ancestor with multiple depth-3 children on canonical surface | `COLLISION_AMBIGUOUS` (WARNING) | Must pick one |
| Canonical branch has no intersection with canonical surface | `COLLISION_CROSS_BRANCH` (WARNING) | Must remap |
| Identification failed | `UNRESOLVED` (ERROR) | Must map or suppress |


In [ ]:
# Load and predict on a small subset
subset = dataset[:100]
wrapped_analyzer = PresidioAnalyzerWrapper(analyzer_engine=analyzer, language='en')
results_df = wrapped_analyzer.predict_dataset(subset)

# Create mapper — no constructor arguments needed
mapper = CanonicalMapper()

# analyze() runs both phases: Identify and Project
mapper.analyze(results_df)

# render_html() shows a color-coded audit table
# (falls back to plain text outside Jupyter)
mapper.render_html()


## 3. All Six Issue Types

After `analyze()`, `get_issues()` returns `MappingIssue` objects sorted by severity
(ERROR first, then WARNING, then INFO), then by token count (most frequent first).

| Issue Type | Severity | Blocking? | When it occurs |
|------------|----------|-----------|----------------|
| `UNRESOLVED` | ERROR | Yes | Label matched nothing in the hierarchy |
| `COLLISION_AMBIGUOUS` | WARNING | Yes | Depth-2 label has multiple depth-3 canonical surface matches |
| `COLLISION_CROSS_BRANCH` | WARNING | Yes | Label maps to a branch absent from the canonical surface |
| `PREDICTION_ONLY` | WARNING | Yes | Label appears only in predictions, never in annotations |
| `COLLISION_TRIVIAL` | INFO | No | Auto-fixed: label collapsed to its sole ancestor on canonical surface |
| `DATASET_ONLY` | INFO | No | Label appears only in annotations, never in predictions |

**Blocking** means `get_mapped_results_dataframe()` raises `IncompleteMapping` until resolved.
INFO issues are non-blocking — they are informational only.


In [ ]:
# View all detected issues with full detail
print(f'Total issues: {len(mapper.get_issues())}')
for issue in mapper.get_issues():
    blocking = 'BLOCKING' if issue.severity != IssueSeverity.INFO else 'non-blocking'
    print(f'  [{issue.severity.value.upper():7}] {issue.type.value:30} | {blocking} | {issue.labels}')
    if issue.annotation_count or issue.prediction_count:
        print(f'             annotations={issue.annotation_count}, predictions={issue.prediction_count}')


### Resolving Issues

Use `map({label: canonical})` to remap a label to a canonical entity,
or `map({label: None})` to suppress it (both annotation and prediction sides become `'O'`).

**When to suppress (`None`)**:
- The model predicts entity types your dataset does not cover (`PREDICTION_ONLY`)
- A label is noise or irrelevant to your evaluation

**When to remap**:
- `COLLISION_AMBIGUOUS`: pick the depth-3 canonical entity that best matches the predictions
- `UNRESOLVED`: manually specify what the label should map to
- `COLLISION_CROSS_BRANCH`: remap to a canonical entity in the correct branch

`map()` validates all targets atomically — if any target is invalid, none are applied.
`map()` returns `self` so you can chain calls.


In [ ]:
# Programmatic batch resolution — iterate over issues by type

# PREDICTION_ONLY: suppress labels the model predicts but dataset never annotates
prediction_only = [i for i in mapper.get_issues() if i.type == IssueType.PREDICTION_ONLY]
for issue in prediction_only:
    print(f'Suppressing PREDICTION_ONLY: {issue.labels}')
    mapper.map({lbl: None for lbl in issue.labels})

# COLLISION_AMBIGUOUS: Presidio depth-2 labels — pick the most appropriate depth-3 target
ambiguous = [i for i in mapper.get_issues() if i.type == IssueType.COLLISION_AMBIGUOUS]
for issue in ambiguous:
    for lbl in issue.labels:
        if lbl == 'PERSON':
            mapper.map({'PERSON': 'NAME'})
        elif lbl == 'DATE_TIME':
            mapper.map({'DATE_TIME': 'DATE'})
        elif lbl == 'LOCATION':
            mapper.map({'LOCATION': 'LOC'})
        elif lbl == 'ORGANIZATION':
            mapper.map({'ORGANIZATION': 'ORG'})
        else:
            # Suppress unknown ambiguous labels
            mapper.map({lbl: None})

# Suppress other known noise
mapper.map({'TITLE': None})
mapper.map({'COMPOSER': None})

# Re-analyze to refresh issue detection
mapper.analyze(results_df)
mapper.render_html()


In [ ]:
# Check remaining blocking issues
blocking = [i for i in mapper.get_issues() if i.severity != IssueSeverity.INFO]
if blocking:
    print(f'{len(blocking)} blocking issue(s) remain:')
    for issue in blocking:
        print(f'  [{issue.severity.value}] {issue.type.value}: {issue.labels}')
else:
    print('No blocking issues - ready to call get_mapped_results_dataframe()')

# DATASET_ONLY issues are INFO (non-blocking) - they tell you what the model never predicts
dataset_only = [i for i in mapper.get_issues() if i.type == IssueType.DATASET_ONLY]
if dataset_only:
    all_ds_only = [lbl for i in dataset_only for lbl in i.labels]
    print(f'\nDataset-only entities (INFO, non-blocking): {all_ds_only}')
    print('  These entities never appear in model predictions — recall for them will be 0.')


## 4. Single Entity Lookup

Use `get_mapping(entity='...')` to look up the projected value for any label,
including labels not yet analyzed. Raises `ValueError` for unresolvable labels.


In [ ]:
# Look up individual entities
print('EMAIL_ADDRESS ->', mapper.get_mapping(entity='EMAIL_ADDRESS'))
print('B-PERSON ->', mapper.get_mapping(entity='B-PERSON'))  # BIO prefix stripped
print('GERMANY_PASSPORT_NUMBER ->', mapper.get_mapping(entity='GERMANY_PASSPORT_NUMBER'))  # COUNTRY tier
print('US_SSN ->', mapper.get_mapping(entity='US_SSN'))

# Full mapping dict for all labels seen so far
full_mapping = mapper.get_mapping()
print('\nFull mapping:')
pprint(full_mapping, compact=True)


## 5. Majority-Vote Depth Auto-Discovery

The canonical surface is computed automatically from the **annotation** labels in the results DataFrame.
Each annotation label is resolved to a canonical entity, its depth in the hierarchy is measured
(capped at 3), and a weighted majority vote determines the canonical depth.

| Scenario | Majority depth | Canonical surface |
|----------|---------------|-------------|
| Most annotations are `NAME`, `EMAIL_ADDRESS`, `SSN` (depth 3) | 3 | depth-3 entities |
| Most annotations are `PERSON`, `LOCATION` (depth 2) | 2 | depth-2 entities |

The canonical surface is **locked** after the first `analyze()` call and does not change on subsequent
calls. This ensures consistent evaluation across multiple models on the same dataset.


In [ ]:
# Demonstrate canonical depth auto-discovery
print(f'Canonical depth (inferred from dataset): {mapper._canonical_depth}')
print(f'Canonical surface size: {len(mapper.canonical_surface)} entities')
print(f'Sample canonical surface entities: {sorted(mapper.canonical_surface)[:10]}')

# EntityHierarchy.get_depth() shows where each entity sits
h = EntityHierarchy(canonical_depth=10)
for entity in ['PERSON', 'NAME', 'FIRST_NAME', 'EMAIL_ADDRESS', 'SSN']:
    try:
        depth = h.get_depth(entity)
        branch = h.canonical_to_branch.get(entity, [])
        print(f'  {entity:25} depth={depth}  branch={" → ".join(branch)}')
    except Exception:
        print(f'  {entity:25} not in hierarchy')


In [ ]:
# Demonstrate projection rules
# After analyze(), inspect _records to see tier and projection_type per label
print(f'{"Label":25} {"Tier":15} {"Canonical":20} {"Projected":20} {"Proj Type":20}')
print('-' * 100)
for lbl, rec in sorted(mapper._records.items()):
    print(f'{lbl:25} {rec.tier:15} {str(rec.canonical):20} {str(rec.projected):20} {str(rec.projection_type):20}')


In [ ]:
# Adjusting the fuzzy threshold: lower = more aggressive auto-resolution
loose_mapper = CanonicalMapper(fuzzy_threshold=0.65)
loose_mapper.analyze(results_df)
print(f'Pending with threshold=0.65: {loose_mapper.pending}')
loose_mapper.render_html()


## 6. Interactive Resolution with resolve_interactively()

`resolve_interactively()` prompts you for each WARNING+ issue one at a time.
It accepts a `prompt_fn` argument for testing (default: `input`).
The interactive session is most useful in a terminal or a Jupyter cell — it
walks you through every WARNING/ERROR issue and lets you type a canonical name
or `None` to suppress.

```python
# Interactive resolution in a terminal or Jupyter cell
# (commented out here to avoid blocking automated notebook runs)
# mapper.resolve_interactively()
```

For automated / batch resolution, use `map()` directly as shown in section 3.


In [ ]:
# Get the full mapping dict once no blocking issues remain
if mapper.pending:
    print(f'Still {len(mapper.pending)} pending: {mapper.pending}')
else:
    entity_mapping = mapper.get_mapping()
    print('Entity mapping ready:')
    pprint(entity_mapping, compact=True)

# Get the remapped DataFrame (includes original_annotation / original_prediction columns)
try:
    mapped_df = mapper.get_mapped_results_dataframe()
    print(f'\nMapped DataFrame: {mapped_df.shape[0]} rows, {mapped_df.shape[1]} columns')
    print('Columns:', list(mapped_df.columns))
except IncompleteMapping as e:
    print(f'Blocked: {e}')


In [13]:
## 7. Eval Surface Locking for Multi-Model Comparison

The canonical surface is locked after the first `analyze()` call. When you call `analyze()`
again (e.g., with a second model's results), the canonical surface stays the same.
This guarantees that both models are evaluated on identical entity types.


Entity mapping (19 labels):
{'AGE': 'AGE',
 'COMPOSER': None,
 'CREDIT_CARD': 'FINANCIAL',
 'DATE_TIME': 'DATE_TIME',
 'DOMAIN_NAME': 'DOMAIN',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'GPE': 'GPE',
 'IBAN_CODE': 'FINANCIAL',
 'LOCATION': 'LOCATION',
 'NRP': 'NATIONALITY',
 'ORGANIZATION': None,
 'PERSON': 'PERSON',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'STREET_ADDRESS': 'ADDRESS',
 'TITLE': None,
 'URL': 'URL',
 'US_DRIVER_LICENSE': 'DRIVER_LICENSE',
 'US_SSN': 'SSN',
 'ZIP_CODE': 'ADDRESS'}

Model entities to evaluate (14):
['ADDRESS', 'AGE', 'DATE_TIME', 'DOMAIN', 'DRIVER_LICENSE', 'EMAIL_ADDRESS',
 'FINANCIAL', 'GPE', 'LOCATION', 'NATIONALITY', 'PERSON', 'PHONE_NUMBER', 'SSN',
 'URL']


In [ ]:
# Demonstrate eval-surface locking across two models
comparison_mapper = CanonicalMapper()

# Model 1
comparison_mapper.analyze(results_df)
surface_after_model1 = frozenset(comparison_mapper.canonical_surface)
print(f'Canonical surface after model 1: {len(surface_after_model1)} entities')

# Model 2 (using same results_df for illustration; in practice this would be a different model)
comparison_mapper.analyze(results_df)
surface_after_model2 = frozenset(comparison_mapper.canonical_surface)
print(f'Canonical surface after model 2: {len(surface_after_model2)} entities')

assert surface_after_model1 == surface_after_model2, 'Canonical surface changed - this should not happen!'
print('Canonical surface is identical across both models. Multi-model comparison is consistent.')


In [ ]:
## 8. EntityHierarchy Customisation

Pass a custom hierarchy dict or `EntityHierarchy` instance to `CanonicalMapper(hierarchy=...)` to:
- Add aliases for labels not in the default taxonomy
- Extend the hierarchy with new entity types
- Override the default mappings


## Summary

### Key Takeaways

1. **Two-phase workflow**: `analyze()` runs Identify (5 tiers) then Project (majority-vote depth)

2. **Canonical depth is data-driven**: computed from annotation label depths; locked after first `analyze()`

3. **Six issue types**:
   - ERROR (blocking): `UNRESOLVED`
   - WARNING (blocking): `COLLISION_AMBIGUOUS`, `COLLISION_CROSS_BRANCH`, `PREDICTION_ONLY`
   - INFO (non-blocking): `COLLISION_TRIVIAL`, `DATASET_ONLY`

4. **Resolution options**:
   - `map({label: canonical})` — remap
   - `map({label: None})` — suppress
   - `resolve_interactively()` — guided interactive session

5. **Canonical surface locking** ensures consistent multi-model comparison

6. **`get_mapped_results_dataframe()`** adds `original_annotation` and `original_prediction` columns
   preserving raw labels before remapping

### Best Practices

1. Call `render_html()` after `analyze()` to visually audit all mappings
2. Pre-map known overrides *before* `analyze()` to keep the issue list focused
3. INFO issues (`DATASET_ONLY`, `COLLISION_TRIVIAL`) are non-blocking — review but no action required
4. Use the same mapper instance across multiple models to ensure canonical surface consistency


In [ ]:
print('''
=== CanonicalMapper Quick Reference ===

# 1. Create (no constructor args needed)
mapper = CanonicalMapper()

# 2. Pre-map known labels before analyze() (optional)
mapper.map({'MY_LABEL': 'PERSON', 'NOISE': None})

# 3. Analyze — Phase 1 (Identify) + Phase 2 (Project)
mapper.analyze(results_df)   # canonical depth inferred from annotations

# 4. Review
mapper.render_html()          # HTML audit table (Jupyter)
mapper.get_issues()           # structured MappingIssue list
mapper.pending                # UNRESOLVED-tier labels

# 5. Resolve blocking issues (WARNING+)
mapper.map({'PERSON': 'NAME'})             # remap COLLISION_AMBIGUOUS
mapper.map({'PREDICTION_ONLY_LBL': None})  # suppress PREDICTION_ONLY
# mapper.resolve_interactively()           # guided interactive session

# 6. Single entity lookup
mapper.get_mapping(entity='B-PERSON')   # strips BIO, returns projected value

# 7. Get results
entity_mapping = mapper.get_mapping()         # full {label: projected} dict
mapped_df = mapper.get_mapped_results_dataframe()  # raises IncompleteMapping if blocking

# 8. Evaluate
evaluator = SpanEvaluator(iou_threshold=0.75)
scores = evaluator.calculate_score_on_df(results_df=mapped_df)
''')
